# Week 12: Evaluation — Recall, Precision, and Groundedness

Same logic as `evaluation_report.py`. Requires `LLM_API_KEY` and `LLM_MODEL` in a `.env` file (see `.env.example`) — used for generating each answer and for an independent groundedness judge call. Retrieval metrics need no API key. Reuses the persistent collection Week 9 built (`data/processed/chroma`) — run `examples/week-09/build_passage_index.py` first if you haven't already.

Verified for real before this notebook was written: mean recall@3 is **93.3%** and mean precision@3 is **31.1%** on `data/sample/eval_questions.json`'s 15 questions — retrieving more (k=3) improves recall as a safety net but dilutes precision, exactly as information-retrieval theory predicts.

In [ ]:
import json
from functools import partial
from pathlib import Path

from ai_finance_course.evaluation import check_groundedness, evaluate_retrieval
from ai_finance_course.rag import answer_question
from ai_finance_course.vector_store import get_or_create_collection, query_collection

QUESTIONS_PATH = Path("data/sample/eval_questions.json")
PERSIST_PATH = Path("data/processed/chroma")
COLLECTION_NAME = "sample_passages"

questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
collection = get_or_create_collection(PERSIST_PATH, COLLECTION_NAME)
len(questions)

## Retrieval Metrics: Recall@3 and Precision@3

In [ ]:
retrieval_result = evaluate_retrieval(
    questions, partial(query_collection, collection, n_results=3), k=3
)
print(f"Mean recall@3:    {retrieval_result['mean_recall_at_k']:.1%}")
print(f"Mean precision@3: {retrieval_result['mean_precision_at_k']:.1%}")

## Which Questions Does Retrieval Miss?

In [ ]:
retrieval_failures = [r for r in retrieval_result["per_question"] if r["recall"] < 1.0]
for failure in retrieval_failures:
    print(f"recall={failure['recall']:.2f} precision={failure['precision']:.2f}  {failure['query']!r}")
if not retrieval_failures:
    print("No retrieval failures at k=3.")

## Set Up the Real LLM Call

In [ ]:
import os

import httpx
from dotenv import load_dotenv

ANTHROPIC_MESSAGES_URL = "https://api.anthropic.com/v1/messages"


def call_llm(prompt: str) -> str:
    """The one Anthropic-specific piece; answer_question() and check_groundedness() are provider-agnostic."""
    with httpx.Client(timeout=60.0) as client:
        response = client.post(
            ANTHROPIC_MESSAGES_URL,
            headers={
                "x-api-key": os.environ["LLM_API_KEY"],
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": os.environ["LLM_MODEL"],
                "max_tokens": 1024,
                "messages": [{"role": "user", "content": prompt}],
            },
        )
        response.raise_for_status()
        data = response.json()
        for block in data["content"]:
            if block["type"] == "text":
                return block["text"]
        raise ValueError(f"No text block in response: {data}")


load_dotenv()

## Generate Answers and Check Groundedness

In [ ]:
groundedness_failures = []
for question in questions:
    result, evidence = answer_question(question["query"], collection, call_llm, n_results=3)
    cited_texts = [evidence[c - 1]["text"] for c in result.citations]
    check = check_groundedness(result.answer, cited_texts, call_llm)
    status = "OK  " if check.grounded else "FAIL"
    print(f"{status} {question['query']!r}")
    print(f"      answer: {result.answer}")
    print(f"      grounded: {check.grounded} — {check.reasoning}")
    if not check.grounded:
        groundedness_failures.append({"query": question["query"], "reasoning": check.reasoning})

## Summary

In [ ]:
print(f"Retrieval failures: {len(retrieval_failures)}/{len(questions)}")
print(f"Groundedness failures: {len(groundedness_failures)}/{len(questions)}")